# Group 12 — PLSA Information Retrieval Project (All 20 Categories)
This notebook trains PLSA on **all 20 categories** of 20 Newsgroups. It implements PLSA with NumPy/EM, inspects topics, tests controlled mixtures from two selected categories, performs query folding-in, and compares retrieval against TF-IDF/VSM and LSI.

**Main experiment:** all 20 categories. **Boundary experiment:** a controlled mixture of two source categories (baseball and politics), as specified in the proposal. Category labels are not used to train PLSA; they are used for evaluation and selecting synthetic source documents.


## 0. Install dependencies

Run once in a notebook environment if required. In Colab, use the first line in a code cell.

In [ ]:
# %pip -q install numpy scipy scikit-learn pandas matplotlib

## 1. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)


## 2. Load the selected dataset

We load only the two categories selected for the topic-boundary experiment. The category labels are used for evaluation and for choosing source documents for synthetic mixtures; PLSA itself is unsupervised and does not receive these labels during training.

In [ ]:
dataset = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=SEED
)
documents = np.array(dataset.data, dtype=object)
labels = np.asarray(dataset.target)
target_names = list(dataset.target_names)

print("Number of categories:", len(target_names))
print("Total documents:", len(documents))
for i, name in enumerate(target_names):
    print(f"{i:2d} | {name:25s} | {(labels == i).sum():5d} documents")

### Main corpus versus boundary experiment

The model and retrieval evaluation use the **full 20-category corpus**. The controlled boundary experiment still uses two chosen source categories to create interpretable 90:10 through 10:90 word mixtures. This does not mean the PLSA model was trained on only those two categories.

## 3. Train/test split and preprocessing

Split the raw documents first. Fit the vocabulary only on the training split to avoid vocabulary leakage from the test documents.

In [ ]:
import re
import spacy
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

docs_train, docs_test, y_train, y_test = train_test_split(
    documents,
    labels,
    test_size=0.25,
    random_state=SEED,
    stratify=labels
)
nlp = spacy.load(
    "en_core_web_sm",
    disable=["ner", "parser"]
)
def preprocess_texts(texts):
    """
    Lowercase, tokenize, remove stop words and punctuation,
    and lemmatize the remaining tokens.

    Returns a list of preprocessed document strings.
    """
    processed_docs = []

    for doc in nlp.pipe(texts, batch_size=256):
        tokens = []

        for token in doc:

            if (
                token.is_space
                or token.is_punct
                or token.like_num
                or not token.is_alpha
            ):
                continue


            if token.is_stop:
                continue


            lemma = token.lemma_.lower().strip()


            if len(lemma) < 2:
                continue

            tokens.append(lemma)

        processed_docs.append(" ".join(tokens))

    return processed_docs
docs_train_processed = preprocess_texts(docs_train)
docs_test_processed = preprocess_texts(docs_test)
count_vectorizer = CountVectorizer(
    lowercase=False,
    tokenizer=str.split,
    token_pattern=None,
    max_features=3000,
    min_df=3,
    max_df=0.90,
    dtype=np.float64
)
X_train = count_vectorizer.fit_transform(
    docs_train_processed
)
X_test = count_vectorizer.transform(
    docs_test_processed
)
vocabulary = count_vectorizer.get_feature_names_out()
print("X_train:", X_train.shape, "(documents, vocabulary)")
print("X_test: ", X_test.shape)
print("Example vocabulary:", vocabulary[:20])

print(
    "Empty training documents after preprocessing:",
    sum(not doc.strip() for doc in docs_train_processed)
)

print(
    "Empty test documents after preprocessing:",
    sum(not doc.strip() for doc in docs_test_processed)
)


## 4. PLSA theory

For document \(d\), word \(w\), and latent topic \(z\):

\[
P(w\mid d)=\sum_{z=1}^{K}P(w\mid z)P(z\mid d)
\]

- \(\theta_{d,z}=P(z\mid d)\): document-topic distribution.
- \(\phi_{z,w}=P(w\mid z)\): topic-word distribution.
- Each row of \(\theta\) and \(\phi\) sums to one.

For every observed document-word pair, the E-step computes the posterior responsibility:

\[
P(z\mid d,w)=\frac{P(z\mid d)P(w\mid z)}
{\sum_{z'}P(z'\mid d)P(w\mid z')}
\]

The M-step uses the expected word counts to update both distributions. The process repeats until the log-likelihood stops improving.